# Lecture 03 (2/2): Integer Programming

# Linear Menu

In [1]:
# linearmenu.py
# p.104, Equation 4.2
import pulp
import random

from itertools import product

commands = ['c0', 'c1', 'c2', 'c3']
frequencies = {
    'c0': 10,
    'c1': 5,
    'c2': 3,
    'c3': 2
}

slots = [0, 1, 2, 3]

cs = list(product(commands, slots))
costs = {(command, slot): frequencies[command] * slot for command, slot in cs}

prob = pulp.LpProblem('LinearMenu-1', pulp.LpMinimize)

x = pulp.LpVariable.dicts('x', cs, cat='Binary')

# Each slot must be filled by exactly one command
for s in slots:
    prob += pulp.lpSum([x[c, s] for c in commands]) == 1

# Each command must be assigned to exactly one slot
for c in commands:
    prob += pulp.lpSum([x[c, s] for s in slots]) == 1

# Objective function
objective = pulp.lpSum([costs[command, slot] * x[command, slot] for command, slot in cs])
prob += objective

status = prob.solve()
print(pulp.LpStatus[status])

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/kotarohara/repo/Python/web-optimization/venv/lib/python3.9/site-packages/pulp/apis/../solverdir/cbc/osx/64/cbc /var/folders/4_/vrr8kzqn5b9dxsprxn13022m0000gn/T/1e6eda19a5d742ac80083beec9ce5ae0-pulp.mps timeMode elapsed branch printingOptions all solution /var/folders/4_/vrr8kzqn5b9dxsprxn13022m0000gn/T/1e6eda19a5d742ac80083beec9ce5ae0-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 13 COLUMNS
At line 90 RHS
At line 99 BOUNDS
At line 116 ENDATA
Problem MODEL has 8 rows, 16 columns and 32 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 17 - 0.00 seconds
Cgl0004I processed model has 8 rows, 16 columns (16 integer (16 of which binary)) and 32 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0038I Initial state - 0 integers unsatisfied sum - 0
Cbc0038I Solution found of 17
Cbc0

# Team Formation
Let's consider an optimization problem where we want to form pairs of students for a team assignment. Imagine that we have 20 students in a class. Half of the students are in a master's program and half are in a PhD program. We want to form as many pairs as possible. Note,

* Some students already formed pairs and they have partners
* Some students prefer working alone
* We want to encourage master's students and PhD students to work together.

In [2]:
import pulp

students = [
    { "id": 0, "program": "master" },
    { "id": 1, "program": "master" },
    { "id": 2, "program": "master" },
    { "id": 3, "program": "master" },
    { "id": 4, "program": "master" },
    { "id": 5, "program": "master" },
    { "id": 6, "program": "master" },
    { "id": 7, "program": "master" },
    { "id": 8, "program": "master" },
    { "id": 9, "program": "master" },
    { "id": 10, "program": "phd" },
    { "id": 11, "program": "phd" },
    { "id": 12, "program": "phd" },
    { "id": 13, "program": "phd" },
    { "id": 14, "program": "phd" },
    { "id": 15, "program": "phd" },
    { "id": 16, "program": "phd" },
    { "id": 17, "program": "phd" },
    { "id": 18, "program": "phd" },
    { "id": 19, "program": "phd" },
]
sids = [s["id"] for s in students]
print(f"Students: {sids}")

from itertools import product
ss = list(product(sids, sids))
print(f"Pairs: {ss[:10]}...")

Students: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Pairs: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9)]...


## Variables and Domains
We use a Python library called `pulp` to solve this integer programming problem. The first step is to define a problem. Then for the given problem, specify variables and their domains.

In [3]:
prob = pulp.LpProblem('TeamMaking', pulp.LpMaximize)


In [4]:
prob

TeamMaking:
MAXIMIZE
None
VARIABLES

In [6]:
ss[:10]

[(0, 0),
 (0, 1),
 (0, 2),
 (0, 3),
 (0, 4),
 (0, 5),
 (0, 6),
 (0, 7),
 (0, 8),
 (0, 9)]

In [7]:
pair = pulp.LpVariable.dicts('pair', ss, cat='Binary')

## Constraints
We set a couple of constraints.

In [8]:
# Each student can only find one partner
for sid in sids:
    prob += pulp.lpSum([pair[sid, sid2] for sid2 in sids]) == 1

# The partnership needs to be symmetric
for _ss in ss:
    prob += pair[_ss[0], _ss[1]] - pair[_ss[1], _ss[0]] == 0

# Some students have already formed a pair
fixed_pairs = [(2, 3), (4, 5), (6, 7)]
for fp in fixed_pairs:
    prob += pair[fp[0], fp[1]] == 1

## Objective
In addition to the constraints, let's add objective. Let's say we want to encourage a PhD student and a non-PhD student to work together.

In [9]:
# Preference
def weight(sid1, sid2):
    reward = 0

    # It is nice if phd student and non-phd student work together
    program1 = students[sid1]['program']
    program2 = students[sid2]['program']
    if program1 != program2:
        reward += 1

    # It is nice if students work together with another student
#     if sid1 != sid2:
#         reward += 1

    return reward

In [10]:
prob += pulp.lpSum([pair[sid1, sid2] * weight(sid1, sid2) for sid1, sid2 in ss])

## Solve the problem

In [11]:
status = prob.solve()
pulp.LpStatus[status]

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/kotarohara/repo/Python/web-optimization/venv/lib/python3.9/site-packages/pulp/apis/../solverdir/cbc/osx/64/cbc /var/folders/4_/vrr8kzqn5b9dxsprxn13022m0000gn/T/088fd5a2b2f841b288fb83e076c1cca8-pulp.mps max timeMode elapsed branch printingOptions all solution /var/folders/4_/vrr8kzqn5b9dxsprxn13022m0000gn/T/088fd5a2b2f841b288fb83e076c1cca8-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 428 COLUMNS
At line 2612 RHS
At line 3036 BOUNDS
At line 3437 ENDATA
Problem MODEL has 423 rows, 400 columns and 1163 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 8 - 0.00 seconds
Cgl0004I processed model has 14 rows, 105 columns (105 integer (105 of which binary)) and 196 elements
Cutoff increment increased from 1e-05 to 1.9999
Cbc0038I Initial state - 0 integers unsatisfied sum - 0
Cbc0038I Soluti

'Optimal'

In [12]:
for key in ss:
    if pair[key].value() > 0:
        print(pair[key])
        print(students[key[0]])
        print(students[key[1]])
        print()

pair_(0,_10)
{'id': 0, 'program': 'master'}
{'id': 10, 'program': 'phd'}

pair_(1,_13)
{'id': 1, 'program': 'master'}
{'id': 13, 'program': 'phd'}

pair_(2,_3)
{'id': 2, 'program': 'master'}
{'id': 3, 'program': 'master'}

pair_(3,_2)
{'id': 3, 'program': 'master'}
{'id': 2, 'program': 'master'}

pair_(4,_5)
{'id': 4, 'program': 'master'}
{'id': 5, 'program': 'master'}

pair_(5,_4)
{'id': 5, 'program': 'master'}
{'id': 4, 'program': 'master'}

pair_(6,_7)
{'id': 6, 'program': 'master'}
{'id': 7, 'program': 'master'}

pair_(7,_6)
{'id': 7, 'program': 'master'}
{'id': 6, 'program': 'master'}

pair_(8,_11)
{'id': 8, 'program': 'master'}
{'id': 11, 'program': 'phd'}

pair_(9,_17)
{'id': 9, 'program': 'master'}
{'id': 17, 'program': 'phd'}

pair_(10,_0)
{'id': 10, 'program': 'phd'}
{'id': 0, 'program': 'master'}

pair_(11,_8)
{'id': 11, 'program': 'phd'}
{'id': 8, 'program': 'master'}

pair_(12,_12)
{'id': 12, 'program': 'phd'}
{'id': 12, 'program': 'phd'}

pair_(13,_1)
{'id': 13, 'program'

In [55]:
for command, slot in cs:
    print(f"command: {command}, slot: {slot}, {x[command, slot].value()}")

command: c0, slot: 0, 1.0
command: c0, slot: 1, 0.0
command: c0, slot: 2, 0.0
command: c0, slot: 3, 0.0
command: c1, slot: 0, 0.0
command: c1, slot: 1, 1.0
command: c1, slot: 2, 0.0
command: c1, slot: 3, 0.0
command: c2, slot: 0, 0.0
command: c2, slot: 1, 0.0
command: c2, slot: 2, 1.0
command: c2, slot: 3, 0.0
command: c3, slot: 0, 0.0
command: c3, slot: 1, 0.0
command: c3, slot: 2, 0.0
command: c3, slot: 3, 1.0


In [1]:
latlngs = [[1.2977225823336935, 103.84952733728228], [1.2973781515324483, 103.84994230125962],[1.297238398446357, 103.84975958502555], [1.2974254695047336, 103.84925656503181]]

In [29]:
lnglats = [[p[1], p[0], 1] for p in latlngs]

In [30]:
xys = [[60,114.66666412353516, 1],[1099,50.66666793823242, 1],[1102,530.6666870117188, 1],[37,411.6666564941406, 1]]

In [63]:
lnglats

[[103.84952733728228, 1.2977225823336935, 1],
 [103.84994230125962, 1.2973781515324483, 1],
 [103.84975958502555, 1.297238398446357, 1],
 [103.84925656503181, 1.2974254695047336, 1]]

In [64]:
xys

[[60, 114.66666412353516, 1],
 [1099, 50.66666793823242, 1],
 [1102, 530.6666870117188, 1],
 [37, 411.6666564941406, 1]]

In [33]:
import numpy as np

In [68]:
# lnglats = A * xys



# https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html

A = np.linalg.lstsq(xys, lnglats)[0]

/var/folders/4_/vrr8kzqn5b9dxsprxn13022m0000gn/T/ipykernel_75578/2464500919.py:9: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  A = np.linalg.lstsq(xys, lnglats)[0]


In [98]:
# https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html
# (lng, lat) = A * (x, y)
xy1 = np.array(xys[0][:2])
ll = np.array(lnglats[0][:2])
A = np.linalg.solve(xys[:2,:2], lnglats[:2, :2])

In [100]:
A@xy1

array([ 3.32012097, 53.90032011])

In [66]:
result = np.dot(A, np.array(xys).T)

In [67]:
result

array([[-2.25033446e-07,  4.83692973e-04,  3.70806373e-04,
        -8.12861025e-05],
       [-8.76148385e-05, -5.96688320e-04, -8.34338491e-04,
        -2.21729114e-04],
       [ 6.38077493e+03,  1.14197358e+05,  1.15131810e+05,
         4.37765735e+03]])

In [60]:
first_point = result[:, 0]

In [61]:
first_point / first_point[2]

array([-3.52674164e-11, -1.37310655e-08,  1.00000000e+00])

In [62]:
A

array([[ 4.51093382e-07, -2.37999741e-07, -4.27573740e-19],
       [-5.20261851e-07, -4.91852867e-07,  2.17690943e-18],
       [ 1.03849506e+02,  1.29771408e+00,  1.00000000e+00]])